# 02_pilot_run: 実データによる Dify 疎通確認

**目的**: 実データ（少数件）を使って、CSV読み込み → split → Dify送信 → 結果取得 までの一連フローが動作することを確認する。

**前提条件**:
- Dify ワークフローが構築・公開済み（`docs/dify/dify_workflow_setup.md` 参照）
- `.env` ファイルに `DIFY_BASE_URL` と `DIFY_API_KEY` が設定済み
- 全テストが通っている状態（`pytest tests/`）

**今回の入力データ**: `data/test_data.csv`（ML 1件、R016 連続マーカー混在の困難ケース）

**確認するポイント**:
1. CSV カラム名のマッピング（実カラム → split.py 期待カラム）
2. split.py の分割結果（R016 は7分割になるはず）
3. Dify API への接続（認証、ネットワーク）
4. LLM応答のパース（JSON配列、件数一致）
5. 分類結果の妥当性（各 sub_id にコードが振られているか）

## セクション 1: セットアップ

In [ ]:
import json
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

# プロジェクトルートからの相対パス
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# .env 読み込み（python-dotenv が必要: pip install python-dotenv）
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
    print("✅ .env loaded")
except ImportError:
    print("⚠️ python-dotenv not installed. 環境変数を直接設定してください")

# モジュール import
from codes_loader import load_codes
from dify_client import (
    DifyClient,
    DifyClientError,
    flatten_results,
    collect_failed_records,
)
from prompt_builder import RECORD_KEY_ORDER
from split import ColumnMapping, SplitConfig, split_records

# ロギング設定（Notebookで見やすく）
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True,  # 既存設定を上書き
)

# pandas表示設定
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)

# 環境変数チェック
base_url = os.environ.get("DIFY_BASE_URL")
api_key = os.environ.get("DIFY_API_KEY")

print(f"DIFY_BASE_URL: {'✅ 設定済み' if base_url else '❌ 未設定'}")
print(f"DIFY_API_KEY: {'✅ 設定済み' if api_key else '❌ 未設定'}")
if api_key:
    # 全部見せると危ないので頭6文字だけ
    print(f"  キー先頭: {api_key[:6]}...")

## セクション 2: CSV 読み込みとカラム名確認

実データのCSVは split.py のデフォルトと違うカラム名を使うので、`ColumnMapping` で対応する。

In [ ]:
# CSV 読み込み
CSV_PATH = PROJECT_ROOT / "data" / "test_data.csv"

raw_df = pd.read_csv(CSV_PATH)
print(f"読み込み件数: {len(raw_df)} 件")
print(f"カラム: {raw_df.columns.tolist()}")
raw_df

### カラム名マッピング

| CSVカラム名 | split.py の期待カラム名 |
|---|---|
| `ID` | `repair_id` |
| `事業コード` | `product_type` |
| `user_comment` | `user_comment` （同じ） |
| `repair_comment` | `repair_comment` （同じ） |
| `Internal_1` | `internal_1` |
| `Internal_2` | `internal_2` |

他のカラム（`date`, `Develop`, `Product_name`, `Serial`, `Garantee`）は今回の処理では使用しない。

In [ ]:
# ColumnMapping で実カラム名を指定
column_mapping = ColumnMapping(
    repair_id="ID",
    product_type="事業コード",
    user_comment="user_comment",
    repair_comment="repair_comment",
    internal_1="Internal_1",
    internal_2="Internal_2",
)

# 必須カラムが揃っているか確認
required = column_mapping.required_columns()
missing = [c for c in required if c not in raw_df.columns]
if missing:
    print(f"❌ 不足カラム: {missing}")
else:
    print(f"✅ 必須カラム揃い")

# 製品種別の分布
print(f"\n製品種別分布:\n{raw_df['事業コード'].value_counts()}")

## セクション 3: 検証件数の絞り込み

今回は1件のみなのでそのまま使うが、件数が多い場合は `df.head(n)` や `df.sample(n)` で絞り込む。

In [ ]:
# 検証用の件数指定（実データが増えた時はここで絞り込み）
MAX_PILOT_RECORDS = 30  # この件数までに制限

if len(raw_df) > MAX_PILOT_RECORDS:
    print(f"⚠️ 全{len(raw_df)}件のうち先頭{MAX_PILOT_RECORDS}件のみ使用")
    pilot_df = raw_df.head(MAX_PILOT_RECORDS).copy()
else:
    pilot_df = raw_df.copy()

# ML のみフィルタ（今回は元々全件MLだが、念のため）
pilot_df = pilot_df[pilot_df["事業コード"] == "ML"].copy()

print(f"検証対象: {len(pilot_df)} 件")
pilot_df[["ID", "事業コード", "user_comment", "repair_comment"]]

## セクション 4: split.py で分割

ユーザコメント・修理者コメントを ①②③ で分割する。R016 は連続マーカー（`①②` と `③④⑤⑥⑦`）を含む困難ケース。

In [ ]:
split_config = SplitConfig(columns=column_mapping)
split_df, split_report = split_records(pilot_df, split_config)

print(split_report.summary())
print()
if split_report.warnings:
    print("警告:")
    for w in split_report.warnings:
        print(f"  - {w}")
    print()

### 分割結果の目視確認

R016 は 7分割されるはず。連続マーカー `①②` が両方の sub_id に同じテキストを複製していること、`③④⑤⑥⑦` も同様であることを確認する。

In [ ]:
# サマリー表示
split_df[[
    "repair_id", "sub_id", "product_type",
    "user_text", "repair_text", "split_info",
]]

In [ ]:
# 1レコードの全カラム詳細
for _, row in split_df.head(3).iterrows():
    print(f"--- {row['repair_id']} sub_id={row['sub_id']} ({row['split_info']}) ---")
    print(f"  user_text       : {row['user_text']!r}")
    print(f"  user_context    : {row['user_context']!r}")
    print(f"  user_postamble  : {row['user_postamble']!r}")
    print(f"  repair_text     : {row['repair_text']!r}")
    print(f"  repair_context  : {row['repair_context']!r}")
    print(f"  repair_postamble: {row['repair_postamble']!r}")
    print(f"  internal_1      : {row['internal_1']!r}")
    print(f"  internal_2      : {row['internal_2']!r}")
    print()

**ここで一時停止**: 分割結果が想定通りであることを確認してから次に進むこと。

確認ポイント:
- [ ] R016 が 7 sub_id に分かれている（split_info=split:7）
- [ ] 連続マーカー `①②` の両方の sub_id で repair_text が同じ（「レンズ側にて対応します。カメラには異常なし」）
- [ ] `③④⑤⑥⑦` の各 sub_id で repair_text が同じ（「ご指摘外の現象」）
- [ ] internal_1, internal_2 が全 sub_id で複製されている

## セクション 5: Dify 接続疎通テスト（1件だけ送る）

本格的にバッチ処理に入る前に、まず1件だけ送って Dify が応答することを確認する。

**確認ポイント**:
- 認証OK（401/403が返らない）
- ネットワーク到達OK
- LLM が JSON配列を返す
- レコード件数が一致

In [ ]:
# クライアント初期化（環境変数から自動取得）
try:
    client = DifyClient(
        timeout_seconds=120,
        max_retries=2,  # パイロット検証なのでリトライ少なめ
    )
    print(f"✅ DifyClient initialized")
    print(f"  base_url: {client.base_url}")
    print(f"  api_key (head): {client.api_key[:6]}...")
except DifyClientError as e:
    print(f"❌ 初期化失敗: {e}")

In [ ]:
# split.py の出力 → Dify用 list[dict]
all_records = [
    {k: row[k] for k in RECORD_KEY_ORDER}
    for _, row in split_df.iterrows()
]

print(f"全レコード数: {len(all_records)}")

# 1件だけ取り出して疎通確認
single_record = all_records[:1]
print(f"\n疎通確認用1件:")
print(json.dumps(single_record, ensure_ascii=False, indent=2))

In [ ]:
# 1件だけ Dify に送信
print("⏳ Dify ワークフローを実行中...")

try:
    classifications, metadata = client.run_workflow(
        records=single_record,
        product_type="ML",
    )
    print(f"\n✅ 疎通成功")
    print(f"  total_tokens: {metadata.get('total_tokens'):,}")
    print(f"  elapsed_time: {metadata.get('elapsed_time'):.2f} 秒")
    print(f"  返却件数:   {len(classifications)} 件")
except DifyClientError as e:
    print(f"\n❌ 疎通失敗")
    print(f"  {e}")
    classifications = None

In [ ]:
# 1件目のレスポンス内容を確認
if classifications:
    print(json.dumps(classifications[0], ensure_ascii=False, indent=2))

**疎通確認できたら次に進む**。

うまくいかなかった場合のチェックリスト:
- 認証エラー: APIキーが正しいか、Dify管理画面で再生成して試す
- タイムアウト: ネットワーク到達性確認、`base_url` 確認
- JSON パース失敗: Dify ワークフローの「ログ」から実際のLLM応答を確認、プロンプト調整
- 件数不一致: LLMが余計な前置きを返している、ワークフローのモデル設定を確認

## セクション 6: 全件をバッチで実行

疎通確認できたので、全レコードを送信する。

In [ ]:
BATCH_SIZE = 10  # 10件単位（今回はR016の7分割のみなので1バッチ）

print(f"⏳ 全 {len(all_records)} レコードを送信中（batch_size={BATCH_SIZE}）...")

try:
    results, report = client.run_batches(
        records=all_records,
        product_type="ML",
        batch_size=BATCH_SIZE,
    )
    print()
    print(report.summary())
except DifyClientError as e:
    print(f"❌ バッチ実行失敗: {e}")
    results = None
    report = None

In [ ]:
# 失敗バッチがあれば確認
if report and report.failed_batches > 0:
    print("⚠️ 失敗バッチあり:")
    for r in results:
        if not r.success:
            print(f"\nBatch {r.batch_index}:")
            print(f"  records: {[rec['repair_id'] for rec in r.records]}")
            print(f"  error: {r.error[:500]}")

## セクション 7: 結果の妥当性チェック

LLM応答の各フィールドが期待形式になっているか、コードが体系内のものか等を確認する。

In [ ]:
# 結果をフラット化
if results:
    classifications = flatten_results(results)
    print(f"分類結果: {len(classifications)} 件")
else:
    classifications = []

In [ ]:
# CodeBook を読み込んでコードの妥当性を検証
codebook = load_codes(PROJECT_ROOT / "config" / "classification_codes.yaml")

validation_issues = []
for c in classifications:
    rid = c.get("repair_id")
    sub = c.get("sub_id")

    user_code = c.get("user_perspective", {}).get("failure_category_code")
    repair_code = c.get("repair_perspective", {}).get("failure_category_code")

    if not codebook.is_valid_failure_code(user_code, "ML"):
        validation_issues.append(
            f"{rid} sub_id={sub}: user_code '{user_code}' は ML コード体系外"
        )
    if not codebook.is_valid_failure_code(repair_code, "ML"):
        validation_issues.append(
            f"{rid} sub_id={sub}: repair_code '{repair_code}' は ML コード体系外"
        )

    repro = c.get("reproduction_status")
    if repro not in codebook.reproduction_statuses:
        validation_issues.append(
            f"{rid} sub_id={sub}: reproduction_status '{repro}' は未定義"
        )

    env_factors = c.get("environment_factors", [])
    for ef in env_factors:
        if ef not in codebook.environment_factors:
            validation_issues.append(
                f"{rid} sub_id={sub}: env_factor '{ef}' は未定義"
            )

if validation_issues:
    print("⚠️ 妥当性問題あり:")
    for issue in validation_issues:
        print(f"  - {issue}")
else:
    print("✅ 全件の分類コードが体系内")

In [ ]:
# 結果の概要表示
if classifications:
    summary_rows = []
    for c in classifications:
        user_p = c.get("user_perspective", {})
        repair_p = c.get("repair_perspective", {})
        summary_rows.append({
            "repair_id": c.get("repair_id"),
            "sub_id": c.get("sub_id"),
            "user_code": user_p.get("failure_category_code"),
            "user_conf": user_p.get("confidence"),
            "repair_code": repair_p.get("failure_category_code"),
            "repair_conf": repair_p.get("confidence"),
            "reproduction": c.get("reproduction_status"),
            "env_factors": ",".join(c.get("environment_factors", [])),
            "perspective_match": (
                user_p.get("failure_category_code") == repair_p.get("failure_category_code")
            ),
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df

In [ ]:
# 1件詳細表示（デバッグ用）
if classifications:
    print("--- 1件目の詳細 ---")
    print(json.dumps(classifications[0], ensure_ascii=False, indent=2))

## セクション 8: 結果の保存

中間結果を Parquet と JSON で保存。後段の `derive_metrics.py` 等で使用。

In [ ]:
# 出力ディレクトリ
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "pilot_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. 分割済み入力（split_df）を保存
if split_df is not None and not split_df.empty:
    split_path = OUTPUT_DIR / f"{timestamp}_split.parquet"
    split_df.to_parquet(split_path, index=False)
    print(f"✅ 分割結果保存: {split_path}")

# 2. 分類結果を JSON で保存（後段モジュールで使う形式）
if classifications:
    classifications_path = OUTPUT_DIR / f"{timestamp}_classifications.json"
    classifications_path.write_text(
        json.dumps(classifications, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"✅ 分類結果保存: {classifications_path}")

# 3. 実行メタ情報を保存
if report:
    meta = {
        "timestamp": timestamp,
        "csv_path": str(CSV_PATH),
        "input_records": int(report.total_input_records),
        "output_records": int(report.total_output_records),
        "total_batches": int(report.total_batches),
        "successful_batches": int(report.successful_batches),
        "failed_batches": int(report.failed_batches),
        "total_tokens": int(report.total_tokens),
        "elapsed_seconds": round(report.total_elapsed_seconds, 2),
        "validation_issues": validation_issues if 'validation_issues' in dir() else [],
    }
    meta_path = OUTPUT_DIR / f"{timestamp}_meta.json"
    meta_path.write_text(
        json.dumps(meta, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"✅ メタ情報保存: {meta_path}")

## セクション 9: 次のアクション

### 疎通確認OKだった場合
- [ ] 件数を増やしてもう一度実行（CSV更新 → セクション3で件数変更）
- [ ] LLMの分類結果を人手レビューして精度を確認
- [ ] 問題なければ `derive_metrics.py` の実装に進む
- [ ] その後 `output_formatter.py` で Tableau 用フォーマット出力

### 問題があった場合
- 分類精度に問題: プロンプト調整 → `prompt_builder.py` で再生成 → Dify貼り付け
- LLMモデル変更検討: gpt-4o → gpt-4-turbo 等（コスト増だが精度向上）
- 分割結果に問題: `01_split_validation.ipynb` で詳細確認
- 環境要因の判定が弱い: 環境要因 keywords をプロンプトで強化

### コスト確認
1件分の `total_tokens` を確認し、本処理（数百〜数千件）のコストを見積もる:
- GPT-4o: 入力 $2.50/1M tokens、出力 $10.00/1M tokens（参考値）
- 1000件処理時の概算: 数百円〜数千円程度の範囲